# Importing libraries

In [1]:
import pandas as pd
import numpy as np
import pathlib
import pickle
from functools import reduce

In [2]:
pd.set_option('display.max_columns', None)

# Importing datasets

In [3]:
# Checking paths
pathlib.Path("../../HCES 2023-24/Python implementation/Data extraction/Population based MPCE/MPCE 2023_24 2Apr25.dta").resolve().exists()

True

In [40]:
# Reading HH-wise MPCE - For Decile-wise classification
path_mpce = "../../HCES 2023-24/Python implementation/Data extraction/Population based MPCE/MPCE 2023_24 2Apr25.dta"
df_mpce = pd.read_stata(path_mpce)
df_mpce = df_mpce.set_index('hhid')
df_mpce = df_mpce[['hh_size', 'sector', 'mpce', 'decile']]
display(df_mpce.head())

## Level 01 file - For State-wise classification
path_lvl01 = "../../HCES 2023-24/Dta raw files//level_01.dta"
df_lvl01 = pd.read_stata(path_lvl01)
df_lvl01 = df_lvl01.set_index('hhid')
df_lvl01 = df_lvl01[['state']]
display(df_lvl01.head())


## Level 11 file - AC stock
path_lvl11 = "../../HCES 2023-24/Dta raw files//level_11.dta"
df_lvl11 = pd.read_stata(path_lvl11)
df_lvl11 = df_lvl11.set_index('hhid')
df_lvl11 = df_lvl11[['has_ac', 'multiplier']]
display(df_lvl11.head())

## Level 13 file - Expenditure on cooling goods
path_lvl13 = "../../HCES 2023-24/Dta raw files/level_13.dta"
df_lvl13 = pd.read_stata(path_lvl13)
df_lvl13 = df_lvl13.loc[df_lvl13['item_code'].isin([580, 581, 44, 588])]

,hh_size,sector,mpce,decile
hhid,,,,
22300101,4.0,2,9407.24,9
22300201,4.0,2,25398.56,10
22300202,2.0,2,11762.50,10
22300203,2.0,2,11579.24,10
22300204,2.0,2,15350.10,10


,state
hhid,
46667201,1
46667301,1
46667302,1
46667303,1
46667304,1


,has_ac,multiplier
hhid,,
22300101,1.0,57436
22300201,1.0,27497
22300202,1.0,27497
22300203,1.0,27497
22300204,1.0,27497


# Data wrangling

In [41]:
df_lvl13.head()

,index,hhid,questionnaire_num,level,item_code,num_firsthand_purchase,is_purchased_onhire,val_firsthand_purchase,repair_cost,num_secondhand_purchase,val_secondhand_purchase,tot_expenditure,multiplier
2,2,22300101,D,13,588,NaN,NaN,NaN,750.0,NaN,NaN,750,57436
20,20,22300101,D,13,580,NaN,NaN,NaN,300.0,NaN,NaN,300,57436
30,30,22300101,D,13,581,NaN,NaN,NaN,1300.0,NaN,NaN,1300,57436
42,42,22300201,D,13,588,NaN,NaN,NaN,800.0,NaN,NaN,800,27497
45,45,22300201,D,13,580,NaN,NaN,NaN,350.0,NaN,NaN,350,27497


In [42]:
# Reshaping level 08 file for firewood and LPG

## Pivoting
df_lvl13 = df_lvl13.pivot_table(index = 'hhid', columns = 'item_code', values = ['num_firsthand_purchase', 'is_purchased_onhire'])

## Converting columns to single level index
df_lvl13.columns = ["_".join(map(str, x)) for x in df_lvl13.columns]

## Changing column names
df_lvl13 = df_lvl13.rename(columns = {'is_purchased_onhire_44': 'is_purchased_onhire_cooler', 'is_purchased_onhire_580': 'is_purchased_onhire_fan', 'is_purchased_onhire_581': 'is_purchased_onhire_ac', 'is_purchased_onhire_588': 'is_purchased_onhire_refrigerator', 
                                      'num_firsthand_purchase_44': 'num_firsthand_purchase_cooler', 'num_firsthand_purchase_580': 'num_firsthand_purchase_fan', 'num_firsthand_purchase_581': 'num_firsthand_purchase_ac', 'num_firsthand_purchase_588': 'num_firsthand_purchase_refrigerator',
                                      })
display(df_lvl13.head())

# Adding market quantity and value
df_lvl13 = df_lvl13.fillna(0)
display(df_lvl13.head())

,is_purchased_onhire_cooler,is_purchased_onhire_fan,is_purchased_onhire_ac,is_purchased_onhire_refrigerator,num_firsthand_purchase_cooler,num_firsthand_purchase_fan,num_firsthand_purchase_ac,num_firsthand_purchase_refrigerator
hhid,,,,,,,,
22301316,NaN,NaN,2.0,NaN,NaN,NaN,1.0,NaN
22302301,NaN,2.0,NaN,NaN,NaN,1.0,NaN,NaN
22302310,NaN,2.0,NaN,NaN,NaN,1.0,NaN,NaN
22302315,NaN,2.0,NaN,NaN,NaN,1.0,NaN,NaN
22302316,NaN,2.0,NaN,NaN,NaN,1.0,NaN,NaN


,is_purchased_onhire_cooler,is_purchased_onhire_fan,is_purchased_onhire_ac,is_purchased_onhire_refrigerator,num_firsthand_purchase_cooler,num_firsthand_purchase_fan,num_firsthand_purchase_ac,num_firsthand_purchase_refrigerator
hhid,,,,,,,,
22301316,0.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0
22302301,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0
22302310,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0
22302315,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0
22302316,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0


In [43]:
# Merging all files

## Checking shapes
print(f"Shape of MPCE file: {df_mpce.shape}")
print(f"Shape of level 01 file: {df_lvl01.shape}")
print(f"Shape of level 11 file: {df_lvl11.shape}")
print(f"Shape of level 13 file: {df_lvl13.shape}")

## Defining merge function
def merge_df (left_df, right_df):
    df = pd.merge(left_df, right_df, how = 'outer', left_index=True, right_index=True, indicator=True)
    print(f"\nSummary from merging dataframes")
    display(df['_merge'].value_counts())
    df = df.drop(labels = '_merge', axis = 1)
    return df.copy()

## Merging
df = reduce(merge_df, [df_mpce, df_lvl01, df_lvl11, df_lvl13])

Shape of MPCE file: (261953, 4)
Shape of level 01 file: (261953, 1)
Shape of level 11 file: (261953, 2)
Shape of level 13 file: (48273, 8)

Summary from merging dataframes


_merge
both          261953
left_only          0
right_only         0
Name: count, dtype: int64


Summary from merging dataframes


_merge
both          261953
left_only          0
right_only         0
Name: count, dtype: int64


Summary from merging dataframes


_merge
left_only     213680
both           48273
right_only         0
Name: count, dtype: int64

In [44]:
df.head()

,hh_size,sector,mpce,decile,state,has_ac,multiplier,is_purchased_onhire_cooler,is_purchased_onhire_fan,is_purchased_onhire_ac,is_purchased_onhire_refrigerator,num_firsthand_purchase_cooler,num_firsthand_purchase_fan,num_firsthand_purchase_ac,num_firsthand_purchase_refrigerator
hhid,,,,,,,,,,,,,,,
22300101,4.0,2,9407.24,9,34,1.0,57436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22300201,4.0,2,25398.56,10,34,1.0,27497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22300202,2.0,2,11762.50,10,34,1.0,27497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22300203,2.0,2,11579.24,10,34,1.0,27497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22300204,2.0,2,15350.10,10,34,1.0,27497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
# Filling NA values with 0

## For hires and first hand purchase count
df.loc[:,'is_purchased_onhire_cooler':] = df.loc[:,'is_purchased_onhire_cooler':].fillna(0)

## For ac count
df['has_ac'] = df['has_ac'].fillna(0)

In [ ]:
# Renaming column values
df['sector'] = df['sector'].replace({1: 'rural', 2: 'urban'})
df['state'] = df['state'].replace({1: 'Jammu & Kashmir', 2: 'Himachal Pradesh', 3: 'Punjab', 4: 'Chandigarh(U.T.)', 5: 'Uttrakhand', 6: 'Haryana', 7: 'Delhi', 8: 'Rajasthan', 9: 'Uttar Pradesh', 10: 'Bihar', 11: 'Sikkim', 12: 'Arunachal Pradesh', 13: 'Nagaland', 14: 'Manipur', 15: 'Mizoram', 16: 'Tripura', 17: 'Meghalaya', 18: 'Assam', 19: 'West Bengal', 20: 'Jharkhand', 21: 'Odisha', 22: 'Chattisgarh', 23: 'Madhya Pradesh', 24: 'Gujarat', 25: 'Dadra & Nagar Haveli', 27: 'Maharashtra', 28: 'Andhra Pradesh', 29: 'Karnataka', 30: 'Goa', 31: 'Lakshadweep (U.T.)', 32: 'Kerala', 33: 'Tamilnadu', 34: 'Puducherry (U.T.)', 35: 'A and N Islands (U.T.)', 36: 'Telangana', 37: 'Ladakh (U.T.)'})

# Labelling codes
df['has_ac'] = df['has_ac'].replace({1: "yes", 0: "no"})
df.loc[:,'is_purchased_onhire_cooler': 'is_purchased_onhire_refrigerator'] = df.loc[:,'is_purchased_onhire_cooler': 'is_purchased_onhire_refrigerator'].astype(str).replace({1: "yes", 2: "no", 0: "not_applicable"})

# Analysis

## Has AC/cooler?

In [56]:
# Across India

## Calculating
trial = df.groupby(['has_ac'], as_index=False).apply(lambda table: pd.Series(
    {'popn_size': table['multiplier'].sum()/100,
     'sample_size': table.shape[0]}
), include_groups = False)
display(trial)

,has_ac,popn_size,sample_size
0,no,2.050072e+08,185099.0
1,yes,8.666390e+07,76854.0


## Number of items purchased

In [67]:
for col in df.loc[:,'num_firsthand_purchase_cooler':].columns:
    print(f"Unique count for col: {col,df[col].unique()}")

Unique count for col: ('num_firsthand_purchase_cooler', array([0., 1., 4., 2.]))
Unique count for col: ('num_firsthand_purchase_fan', array([0., 1., 2., 3., 4., 5., 6., 7., 9.]))
Unique count for col: ('num_firsthand_purchase_ac', array([0., 1., 2.]))
Unique count for col: ('num_firsthand_purchase_refrigerator', array([0., 1., 2.]))


In [88]:
# Firsthand purchase - Onhire vs full

# Duplicate df and replace Nan values
trial = df.copy()

# Sumproduct of consumption (change column names based on appliance)
value_mult = trial.groupby('is_purchased_onhire_ac').apply(lambda table: pd.Series({
    'num_firsthand_purchase_ac': (table['num_firsthand_purchase_ac'] * table['multiplier']).sum()/100
    }), include_groups = False).round(2)

display(value_mult)

,num_firsthand_purchase_ac
is_purchased_onhire_ac,
no,478541.22
not_applicable,0.00
yes,59597.47
